# Rebuttal analyses — EpiClass

Analyses supporting the point-by-point responses to the peer reviewers. Sections follow the rebuttal document (by reviewer and comment number), and each one reproduces the specific numbers or figures cited in that response.

Figures already incorporated into the revised paper or its Quarto supplement (Ext. Fig 8E-F Input-QC correlations, Ext. Fig 5g read-handling variants, Ext. Fig 9C genome browser) are produced elsewhere and are not reproduced here.

## Contents

1. **R1 — Major comment 2** — Cancer status classifier by biomaterial type (Rebuttal Table 1, Rebuttal Figure 1).
2. **R1 — Major comment 2 (cont.)** — UUID-level vs EpiRR-level cross-validation grouping (Rebuttal Table 2, Rebuttal Figure 2).
3. **R1 — Major comment 3** — Biospecimen source overlap between ENCODE and EpiATLAS (58% core / 73% all).
4. **R1 — Major comment 4** — Classifier performance on SHAP-selected regions (Rebuttal Figure 3; also answers R2 Question 3).
5. **R2** — WGBS biospecimen-source coverage in the 16 trained classes (28 / 98 evaluable samples).
6. **R3 — Question 1** — Per-assay vs mixed-assay biospecimen classifier (Rebuttal Figure 4).
7. **Appendix A** — Biospecimen performance by consortium. Addresses an internal review comment (project/lab/consortium as a possible confounder) about the flagship paper, not an official reviewer comment, and is kept separate.
8. **Appendix B** — Bin–bin correlation at 100 kb (median pairwise |Pearson| = 0.616), backing the discussion statement behind Section 4's small SHAP effect sizes. Documented as code only (needs the raw signal `.npz`), not run inline.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# pylint: disable=import-error, redefined-outer-name, use-dict-literal, too-many-lines, too-many-branches, duplicate-code, too-many-nested-blocks, missing-module-docstring
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
from plotly.subplots import make_subplots
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score
from statsmodels.stats.multitest import multipletests

pio.renderers.default = "notebook_connected"

from epiclass.utils.notebooks.paper.paper_utilities import (
    ASSAY,
    ASSAY_ORDER,
    BIOMATERIAL_TYPE,
    CANCER,
    CELL_TYPE,
    IHECColorMap,
    MetadataHandler,
    SplitResultsHandler,
    load_split_predictions,
    pairwise_ttests,
    save_figure,
)

In [ ]:
base_dir = Path.home() / "Projects/epiclass/output/paper"
paper_dir = base_dir
if not paper_dir.exists():
    raise FileNotFoundError(f"Directory {paper_dir} does not exist.")

base_data_dir = base_dir / "data"
base_fig_dir = base_dir / "figures"

In [ ]:
IHECColorMap = IHECColorMap(base_fig_dir)
assay_colors = IHECColorMap.assay_color_map
cell_type_colors = IHECColorMap.cell_type_color_map

In [ ]:
split_results_handler = SplitResultsHandler()

metadata_handler = MetadataHandler(paper_dir)
metadata_v2 = metadata_handler.load_metadata("v2")
metadata_v2_df = metadata_v2.to_df()

In [ ]:
gen_data_dir = base_data_dir / "training_results" / "dfreeze_v2"

data_dir_100kb = gen_data_dir / "hg38_100kb_all_none"
mixed_data_dir = gen_data_dir / "mixed"

cancer_data_dir = data_dir_100kb / f"{CANCER}_1l_3000n" / "10fold-oversampling"

for path in [gen_data_dir, data_dir_100kb, mixed_data_dir, cancer_data_dir]:
    if not path.exists():
        raise FileNotFoundError(f"Directory {path} does not exist.")

In [ ]:
def change_classifier_task_name(all_metrics):
    """Remove "11c" from the classifier task names to match names for all other feature sets."""
    try:
        all_metrics["hg38_100kb_all_none"][ASSAY] = all_metrics["hg38_100kb_all_none"][  # type: ignore
            f"{ASSAY}_11c"
        ]
        del all_metrics["hg38_100kb_all_none"][f"{ASSAY}_11c"]
    except KeyError:
        pass
    return all_metrics

## 1. R1 — Major comment 2: Cancer status vs biomaterial type

**Reviewer comment.** It is unclear whether the cancer-status classifier captures cancer-specific epigenomic signatures or primarily reflects differences between cell lines and primary tissues; since many cell lines are cancer-derived, biomaterial type may act as a confounder. Evaluating cancer-status predictions within each biomaterial category would help clarify this.

**Analysis.** We break down the cancer classifier's 10-fold cross-validation performance by harmonized biomaterial type and compare it against a naive `DummyClassifier` baseline that predicts the majority cancer-status class within each biomaterial type — i.e. the best a classifier could do from biomaterial type alone. A substantial gap above this baseline for the mixed-composition types (cell line, primary cell, primary tissue) indicates genuine cancer-associated signal.

**Rebuttal → notebook** — each cited number or figure, and the cell/output that produces it:
- Rebuttal Table 1 (class proportions per biomaterial type) → the `*_frac` columns of `cancer_composition_df` below.
- Rebuttal Figure 1 (per-biomaterial accuracy / F1 vs naive baseline) → the final `plot_metrics_per_category` figure.

In [ ]:
cancer_results = split_results_handler.obtain_all_feature_set_data(
    parent_folder=gen_data_dir,
    merge_assays=True,
    return_type="split_results",
    include_categories=[CANCER],
    include_sets=["hg38_100kb_all_none"],
    verbose=False,
)

cancer_split_dfs = cancer_results["hg38_100kb_all_none"][CANCER]

In [ ]:
# Load split results
cancer_concat_df_w_meta = load_split_predictions(
    data_dir=cancer_data_dir,
    metadata=metadata_v2,
    metadata_handler=metadata_handler,
    merge_assays=False,
)
print(cancer_concat_df_w_meta.shape)

In [ ]:
cancer_composition_df = (
    cancer_concat_df_w_meta.groupby([BIOMATERIAL_TYPE, "True class"])
    .size()
    .unstack(fill_value=0)
)
cancer_composition_df["total"] = cancer_composition_df.sum(axis=1)
for col in cancer_composition_df.columns[:-1]:  # skip 'total'
    cancer_composition_df[f"{col}_frac"] = (
        cancer_composition_df[col] / cancer_composition_df["total"]
    )

display(cancer_composition_df)

In [ ]:
def prepare_metrics_per_category(
    split_dfs: Dict,
    breakdown_col: str,
    verbose: bool = False,
) -> Dict[str, Dict[str, Dict[str, Dict[str, float]]]]:
    """Compute per-split metrics broken down by breakdown_col.

    Args:
        concat_df_w_meta: Concatenated predictions DataFrame with metadata joined.
        split_dfs: Raw split result DataFrames from read_split_results.
            Format: {split_name: DataFrame}
        breakdown_col: Column name for the category to break down by in metadata.
        verbose: Whether to print progress.

    Returns:
        Dict mapping breakdown_col -> task_category -> split_name -> metric_dict
    """
    metadata_df = metadata_handler.load_metadata("v2").to_df()
    if breakdown_col not in metadata_df.columns:
        raise ValueError(
            f"'{breakdown_col}' not found in metadata. "
            f"Available columns: {list(metadata_df.columns)}"
        )

    breakdown_order = sorted(metadata_df[breakdown_col].dropna().unique())

    md5_per_category = {
        cat: set(metadata_df.loc[metadata_df[breakdown_col] == cat].index)
        for cat in breakdown_order
    }

    if verbose:
        for btype, md5s in md5_per_category.items():
            print(f"  {btype}: {len(md5s)} samples")

    metrics_per_biomaterial = {}
    for btype in breakdown_order:
        if verbose:
            print(f"Computing metrics for biomaterial: {btype}")

        # Wrap in the classifier-level dict that compute_split_metrics expects:
        # {split_name: {classifier_name: filtered_df}}
        filtered_split_dfs = {}
        for split_name, split_df in split_dfs.items():
            filtered = split_df[split_df.index.isin(md5_per_category[btype])]
            if len(filtered) == 0:
                if verbose:
                    print(f"  WARNING: No samples for {btype} in {split_name}")
                continue
            filtered_split_dfs[split_name] = {"NN": filtered}

        if not filtered_split_dfs:
            if verbose:
                print(f"  Skipping {btype}: no samples in any split.")
            continue

        split_metrics = SplitResultsHandler().compute_split_metrics(
            all_split_dfs=filtered_split_dfs,
            split_list=list(filtered_split_dfs.keys()),
        )
        inverted = SplitResultsHandler.invert_metrics_dict(split_metrics)

        # Delete redundant AUC_macro and rename AUC_micro to AUC for clarity
        for split_name, metrics_dict in inverted["NN"].items():
            metrics_dict["AUC"] = metrics_dict.pop("AUC_micro")
            del metrics_dict["AUC_macro"]

        metrics_per_biomaterial[btype] = inverted

    return metrics_per_biomaterial

In [ ]:
cancer_metrics_per_biomaterial = prepare_metrics_per_category(
    split_dfs=cancer_split_dfs,  # type: ignore
    breakdown_col=BIOMATERIAL_TYPE,
    verbose=False,
)

In [ ]:
def plot_metrics_per_category(
    metrics_per_category: Dict[str, Dict[str, Dict[str, Dict[str, float]]]],
    category_label: str,
    naive_metrics_df: pd.DataFrame | None = None,
    naive_category_col: str | None = None,
    metrics_to_plot: List[str] | None = None,
    category_colors: Dict[str, str] | None = None,
    xaxis_label_map: Dict[str, str] | None = None,
    logdir: Path | None = None,
    name: str | None = None,
    title_prefix: str = "Classification metrics",
    filename_prefix: str = "metrics_per_category",
    y_range: Tuple[float, float] | None = None,
    boxpoints: str = "all",
    width: int = 1200,
    height: int = 800,
    scale: int = 1,
) -> None:
    """Plot per-split metrics broken down by an arbitrary category.

    Uses manual x-positioning to avoid Plotly boxmode='group' spacing bugs
    with make_subplots. Naive baseline boxes, when provided, are drawn
    adjacent to the real model boxes with consistent spacing.

    Args:
        metrics_per_category: Dict mapping category_value -> task_category
            -> split_name -> metric_dict.
        category_label: Display name for the breakdown category (used in
            titles and axis labels).
        naive_metrics_df: Optional DataFrame with naive baseline metrics.
            Must contain naive_category_col, 'split', and metric columns.
        naive_category_col: Column name in naive_metrics_df matching the
            breakdown category. Required if naive_metrics_df is provided.
        metrics_to_plot: List of metric keys to plot. Defaults to
            ["Accuracy", "F1_macro", "AUC"].
        category_colors: Optional dict mapping category values to colors.
            Defaults to Plotly qualitative palette.
        xaxis_label_map: Optional dict mapping category values to display labels for x-axis ticks.
        logdir: Directory to save figures. If None, only display.
        name: Optional suffix for titles and filenames.
        title_prefix: Prefix for the figure title.
        filename_prefix: Prefix for saved filenames.
        y_range: Optional y-axis range.
        boxpoints: "all" or "outliers".
        width: Figure width in pixels.
        height: Figure height in pixels.
        scale: Scale factor for PNG export.
    """
    if boxpoints not in ["all", "outliers"]:
        raise ValueError("Invalid boxpoints value.")
    if naive_metrics_df is not None and naive_category_col is None:
        raise ValueError(
            "naive_category_col must be provided when naive_metrics_df is given."
        )

    if metrics_to_plot is None:
        metrics_to_plot = ["Accuracy", "F1_macro", "AUC"]

    category_values = list(metrics_per_category.keys())

    if category_colors is None:
        category_colors = dict(
            zip(
                category_values,
                px.colors.qualitative.Plotly[2 : len(category_values) + 2],
            )
        )

    has_naive = naive_metrics_df is not None
    n_slots = 2 if has_naive else 1
    group_width = n_slots + 0.8  # spacing between category groups
    box_width = 0.6

    x_positions = {cat_val: i * group_width for i, cat_val in enumerate(category_values)}

    reference_cat = next(iter(metrics_per_category))
    classifier_names = list(metrics_per_category[reference_cat].keys())

    for algo in classifier_names:
        fig = make_subplots(
            rows=1,
            cols=len(metrics_to_plot),
            shared_yaxes=True,
            subplot_titles=[m.replace("_", " ") for m in metrics_to_plot],
            x_title=category_label,
            y_title="Metric Value",
            horizontal_spacing=0.05,
        )

        for col_idx, metric in enumerate(metrics_to_plot, start=1):
            for cat_val in category_values:
                color = category_colors.get(cat_val, "grey")
                x_pos = x_positions[cat_val]

                # --- Model box ---
                y_vals = []
                hovertext = []
                if algo in metrics_per_category[cat_val]:
                    split_dict = metrics_per_category[cat_val][algo]
                    for split_name, metric_dict in split_dict.items():
                        if metric not in metric_dict:
                            continue
                        val = metric_dict[metric]
                        if val is None or (isinstance(val, float) and np.isnan(val)):
                            continue
                        y_vals.append(val)
                        hovertext.append(f"{cat_val} - {split_name}: {val:.4f}")

                if y_vals:
                    fig.add_trace(
                        go.Box(
                            x=[x_pos] * len(y_vals),
                            y=y_vals,
                            name=cat_val,
                            boxmean=True,
                            boxpoints=boxpoints,
                            width=box_width,
                            marker=dict(
                                size=4, color=color, line=dict(color="black", width=0.5)
                            ),
                            line=dict(width=1, color="black"),
                            fillcolor=color,
                            opacity=0.7,
                            hovertemplate="%{text}",
                            text=hovertext,
                            legendgroup=cat_val,
                            showlegend=False,
                        ),
                        row=1,
                        col=col_idx,
                    )

                # --- Naive baseline box ---
                if has_naive:
                    cat_naive = naive_metrics_df[
                        naive_metrics_df[naive_category_col] == cat_val
                    ]
                    naive_vals = []
                    naive_hover = []
                    for _, row in cat_naive.iterrows():
                        val = row.get(metric)
                        if val is None or pd.isna(val):
                            continue
                        naive_vals.append(val)
                        naive_hover.append(
                            f"Naive - {cat_val} - {row['split']}: {val:.4f}"
                        )

                    if naive_vals:
                        fig.add_trace(
                            go.Box(
                                x=[x_pos + 1] * len(naive_vals),
                                y=naive_vals,
                                name="Naive baseline",
                                boxmean=True,
                                boxpoints=boxpoints,
                                width=box_width,
                                marker=dict(
                                    size=4,
                                    color="red",
                                    line=dict(color="black", width=0.5),
                                ),
                                line=dict(width=1, color="black"),
                                fillcolor="rgba(255, 0, 0, 0.15)",
                                hovertemplate="%{text}",
                                text=naive_hover,
                                legendgroup="naive_baseline",
                                showlegend=False,
                            ),
                            row=1,
                            col=col_idx,
                        )

            # X-axis ticks centered on each group
            tick_offset = 0.5 if has_naive else 0

            tick_labels = [
                xaxis_label_map.get(cat, cat) if xaxis_label_map else cat
                for cat in category_values
            ]
            tick_labels = [label.replace(" ", "<br>") for label in tick_labels]

            fig.update_xaxes(
                tickmode="array",
                tickvals=[x_positions[cat] + tick_offset for cat in category_values],
                ticktext=tick_labels,
                row=1,
                col=col_idx,
            )

        # --- Legend entries ---
        for cat_val in category_values:
            color = category_colors.get(cat_val, "grey")
            fig.add_trace(
                go.Scatter(
                    x=[None],
                    y=[None],
                    mode="markers",
                    name=cat_val,
                    marker=dict(size=8, color=color),
                    legendgroup=cat_val,
                    showlegend=True,
                )
            )

        if has_naive:
            fig.add_trace(
                go.Box(
                    y=[None],
                    name="Naive baseline",
                    marker=dict(color="red"),
                    line=dict(color="red"),
                    fillcolor="rgba(255, 0, 0, 0.15)",
                    legendgroup="naive_baseline",
                    showlegend=True,
                )
            )

        # --- Layout ---
        title = f"{title_prefix} per {category_label.lower()}"
        if name is not None:
            title += f" - {name}"

        fig.update_layout(
            width=width,
            height=height,
            title=title,
            legend=dict(itemsizing="constant"),
        )

        if y_range is not None:
            fig.update_yaxes(range=y_range)

        if logdir:
            logdir = Path(logdir)
            logdir.mkdir(parents=True, exist_ok=True)
            base_name = f"{filename_prefix}_{algo}"
            if name is not None:
                base_name += f"_{name}"
            save_figure(fig, logdir, base_name, scale=scale)

        fig.show()

In [ ]:
def compute_naive_majority_metrics_per_category(
    split_dfs: Dict[str, pd.DataFrame],
    breakdown_col: str,
    breakdown_order: List[str] | None = None,
    verbose: bool = False,
) -> pd.DataFrame:
    """Compute metrics for a naive majority-class classifier, per category and split.

    Args:
        split_dfs: Raw split result DataFrames from read_split_results.
            Format: {split_name: DataFrame}
        breakdown_col: Metadata column name to break down by.
        breakdown_order: Optional ordered list of category values to include.
            If None, all unique values are used in sorted order.
        verbose: Whether to print progress.

    Returns:
        DataFrame with columns: breakdown_col, split, n_samples, majority_class,
            majority_fraction, Accuracy, F1_macro, AUC_micro, AUC_macro
    """
    metadata_df = metadata_handler.load_metadata("v2").to_df()
    if breakdown_col not in metadata_df.columns:
        raise ValueError(
            f"'{breakdown_col}' not found in metadata. "
            f"Available columns: {list(metadata_df.columns)}"
        )

    if breakdown_order is None:
        breakdown_order = sorted(metadata_df[breakdown_col].dropna().unique())

    md5_per_category = {
        cat: set(metadata_df.loc[metadata_df[breakdown_col] == cat].index)
        for cat in breakdown_order
    }

    rows = []
    for cat in breakdown_order:
        for split_name, split_df in split_dfs.items():
            filtered = split_df[split_df.index.isin(md5_per_category[cat])]
            if len(filtered) == 0:
                continue

            y_true = filtered["True class"]
            n_samples = len(y_true)
            X_dummy = np.zeros((n_samples, 1))

            majority_class_value = y_true.value_counts().idxmax()
            dummy = DummyClassifier(strategy="constant", constant=majority_class_value)
            dummy.fit(X_dummy, y_true)
            y_pred = dummy.predict(X_dummy)

            majority_fraction = (y_true == majority_class_value).mean()
            present_classes = sorted(y_true.unique())

            accuracy = accuracy_score(y_true, y_pred)
            f1_macro = f1_score(
                y_true, y_pred, labels=present_classes, average="macro", zero_division=0
            )

            rows.append(
                {
                    breakdown_col: cat,
                    "split": split_name,
                    "n_samples": n_samples,
                    "majority_class": majority_class_value,
                    "majority_fraction": majority_fraction,
                    "Accuracy": accuracy,
                    "F1_macro": f1_macro,
                }
            )

            if verbose:
                print(
                    f"  {cat} | {split_name}: n={n_samples}, "
                    f"majority={majority_class_value} ({majority_fraction:.1%}), "
                    f"acc={accuracy:.4f}, f1={f1_macro:.4f}"
                )

    return pd.DataFrame(rows)

In [ ]:
naive_metrics_df = compute_naive_majority_metrics_per_category(
    split_dfs=cancer_split_dfs,  # type: ignore
    breakdown_col=BIOMATERIAL_TYPE,
    verbose=False,
)

In [ ]:
plot_metrics_per_category(
    metrics_per_category=cancer_metrics_per_biomaterial,
    category_label="Biomaterial Type",
    naive_metrics_df=naive_metrics_df,
    naive_category_col=BIOMATERIAL_TYPE,
    y_range=(0.34, 1.01),
    metrics_to_plot=["Accuracy", "F1_macro"],
    logdir=Path.home() / "downloads",
    filename_prefix="cancer_metrics_per_biomaterial",
)

## 2. R1 — Major comment 2 (cont.): UUID-level vs EpiRR-level cross-validation grouping

**Context.** While addressing the comment above we identified that cross-validation splits were constrained at the UUID level (all tracks of a dataset kept together) rather than the EpiRR level stated in the submitted manuscript (all datasets of a biological sample kept together). To check whether the looser UUID grouping introduces detectable data leakage, we re-ran the splits at the EpiRR level and compared per-fold performance.

**Analysis.** Welch's t-tests (n = 10 folds per group) compare the UUID-level split (`hg38_100kb_all_none`) against the EpiRR-level split (`hg38_100kb_all_none_EpiRR_split`) for the Assay and Biospecimen classifiers, on Accuracy and macro F1.

**Rebuttal → notebook** — each cited number or figure, and the cell/output that produces it:
- Rebuttal Table 2 (t-tests, all p > 0.1) → `df_tests` below.
- Rebuttal Figure 2 (per-fold Accuracy / F1 for both splits) → the `graph_feature_set_metrics` figures.

In [ ]:
def graph_feature_set_metrics(
    all_metrics: Dict[str, Dict[str, Dict[str, Dict[str, float]]]],
    logdir: Path | str | None = None,
    name: str | None = None,
    feature_set_order: List[str] | None = None,
    y_range: Tuple[float, float] | None = None,
    boxpoints: str = "all",
    width: int = 1200,
    height: int = 1200,
    verbose: bool = False,
) -> None:
    """Graph the metrics for all feature sets.

    Args:
        all_metrics: Dictionary containing all metrics for all feature sets.
            Format: {feature_set: {task_name: {split_name: metric_dict}}}
        logdir: Directory where the figures will be saved. If None, figures are only displayed.
        name: Optional suffix for the figure name.
        feature_set_order: Optional explicit order for the feature-set boxes.
            Feature sets are plotted in this order; any present in ``all_metrics``
            but not listed are appended afterwards in sorted order. Defaults to
            sorted order when None.
        y_range: Optional y-axis range.
        boxpoints: Type of box points to display ("all" or "outliers").
        width: Figure width.
        height: Figure height.
        verbose: Whether to print progress.
    """
    if boxpoints not in ["all", "outliers"]:
        raise ValueError("Invalid boxpoints value.")

    input_y_range = y_range

    if feature_set_order is not None:
        ordered_feature_sets = [fs for fs in feature_set_order if fs in all_metrics]
        ordered_feature_sets += sorted(
            fs for fs in all_metrics if fs not in ordered_feature_sets
        )
    else:
        ordered_feature_sets = sorted(all_metrics)

    reference_hdf5_type = "hg38_100kb_all_none"
    metadata_categories = list(all_metrics[reference_hdf5_type].keys())

    for _, metadata_label in enumerate(metadata_categories):
        if verbose:
            print(f"Graphing category: {metadata_label}")

        category_fig = make_subplots(
            rows=1,
            cols=2,
            shared_yaxes=True,
            subplot_titles=["Accuracy", "F1-score (macro)"],
            x_title="Feature set",
            y_title="Metric value",
        )

        used_resolutions = set()

        for feature_set_name in ordered_feature_sets:
            if verbose:
                print(f"Processing feature set: {feature_set_name}")

            tasks_dicts = all_metrics[feature_set_name]

            task_name = metadata_label
            if "split" in task_name:
                raise ValueError(
                    "'split' key found at depth 1. Wrong metrics dict format."
                )

            try:
                task_dict = tasks_dicts[task_name]
            except KeyError:
                print(f"Skipping {feature_set_name}, no task name {task_name}.")
                continue

            display_name = feature_set_name.replace("_none", "").replace("hg38_", "")

            resolution = display_name.split("_")[0]
            used_resolutions.add(resolution)

            # Accuracy
            metric = "Accuracy"
            y_vals = [task_dict[split][metric] for split in task_dict]
            hovertext = [
                f"{split}: {metrics_dict[metric]:.4f}"
                for split, metrics_dict in task_dict.items()
            ]

            category_fig.add_trace(
                go.Box(
                    y=y_vals,
                    name=display_name,
                    boxmean=True,
                    boxpoints=boxpoints,
                    marker=dict(size=3, color="black"),
                    line=dict(width=1, color="black"),
                    hovertemplate="%{text}",
                    text=hovertext,
                    legendgroup=resolution,
                    showlegend=False,
                ),
                row=1,
                col=1,
            )

            # F1 macro
            metric = "F1_macro"
            y_vals = [task_dict[split][metric] for split in task_dict]
            hovertext = [
                f"{split}: {metrics_dict[metric]:.4f}"
                for split, metrics_dict in task_dict.items()
            ]

            category_fig.add_trace(
                go.Box(
                    y=y_vals,
                    name=display_name,
                    boxmean=True,
                    boxpoints=boxpoints,
                    marker=dict(size=3, color="black"),
                    line=dict(width=1, color="black"),
                    hovertemplate="%{text}",
                    text=hovertext,
                    legendgroup=resolution,
                    showlegend=False,
                ),
                row=1,
                col=2,
            )

        title = f"Neural network performance - {metadata_label}"
        if name is not None:
            title += f" - {name}"

        category_fig.update_layout(
            width=width,
            height=height,
            title=title,
        )

        # # Adjust y-axis range
        if input_y_range is not None:
            category_fig.update_yaxes(range=input_y_range)

        # Save figure
        if logdir:
            base_name = f"feature_set_metrics_{metadata_label}"
            full_name = f"{base_name}_{name}" if name is not None else base_name
            save_figure(category_fig, logdir, full_name)

        category_fig.show()

In [ ]:
compare_split_method = [
    "hg38_100kb_all_none",
    "hg38_100kb_all_none_EpiRR_split",
]

In [ ]:
all_metrics = split_results_handler.obtain_all_feature_set_data(
    parent_folder=mixed_data_dir,
    merge_assays=True,
    return_type="metrics",
    include_sets=compare_split_method,
    include_categories=[ASSAY, CELL_TYPE],
    exclude_names=["16ct", "27ct", "7c", "chip-seq-only"],
    verbose=False,
)

In [ ]:
# Order the metrics
selected_metrics = {
    name: all_metrics[name]  # type: ignore
    for name in compare_split_method
    if name in all_metrics
}

selected_metrics = change_classifier_task_name(selected_metrics)

In [ ]:
df_tests = pairwise_ttests(selected_metrics)
display(df_tests)

In [ ]:
graph_feature_set_metrics(
    all_metrics=selected_metrics,  # type: ignore
    boxpoints="all",
    height=500,
)

## 3. R1 — Major comment 3: Biospecimen source overlap between ENCODE and EpiATLAS

**Reviewer comment.** The manuscript interpreted the low (~10%) biospecimen overlap between ENCODE and EpiATLAS as evidence of generalization, but biologically similar cell types may still dominate both resources. It would help to clarify how distinct biospecimen sources were defined and whether closely related lines are treated as separate.

**Context.** On re-examination the original ~10% figure was computed on an incomplete metadata version and reflected only the proportion of ENCODE samples matching one of the 16 biospecimen-classifier classes — it was mis-stated as the overall overlap. Here we recompute the true ontology-term overlap. The original generalization claim was removed from the manuscript; this code is provided for the reviewer.

**Analysis.** For each ENCODE sample we extract the biospecimen ontology CURIE (Cell Ontology / Uberon / EFO) and test for an exact match against any EpiATLAS `harmonized_sample_ontology_curie`. `encode_df` is already restricted to the evaluable set (no EpiRR shared with EpiATLAS), asserted below.

**Rebuttal → notebook** — each cited number or figure, and the cell/output that produces it:
- 73% overlap for all ENCODE samples (6,386 / 8,777) → the first overlap print.
- 58% overlap restricting to core assays (2,735 / 4,716) → the core-subset overlap print.

Full semantic (Lin) similarity for the non-exact-match samples lives in `evaluate_biospecimen_similarity.ipynb`([permalink](https://github.com/labjacquespe/EpiClass/blob/8fb112bffee22d9a5c995493075ea6847aba46d1/src/python/epiclass/utils/notebooks/paper/evaluate_biospecimen_similarity.ipynb)).

In [ ]:
sf1_df_path = (
    Path.home()
    / "downloads"
    / "SF1_EpiATLAS_merged_pred_results_2.1_chrY_zscores_82cols.csv.xz"
)
sf3_df_path = (
    Path.home()
    / "downloads"
    / "SF3_ENCODE_predictions_merge_metadata_2025-02_freeze1_98cols.csv.xz"
)

epiatlas_df = pd.read_csv(sf1_df_path, low_memory=False)
print(epiatlas_df.shape)

encode_df = pd.read_csv(sf3_df_path, low_memory=False)
print(encode_df.shape)

In [ ]:
assert ~(encode_df["in_epiatlas"]).all()  # no epiatlas overlap, all False

In [ ]:
biosample_ontology_sources = frozenset(("NTR", "EFO", "CL", "UBERON"))
BIOSPECIMEN_ID_COL = "biospecimen_ontology_id"

In [ ]:
def extract_biosample_curie(df: pd.DataFrame) -> pd.Series:
    """
    Extract the ontology CURIE from the ENCODE FILE_biosample_ontology column.

    Examples
    --------
    /biosample-types/primary_cell_CL_1001606/ -> CL:1001606
    /biosample-types/cell_line_EFO_0001086/   -> EFO:0001086
    /biosample-types/tissue_UBERON_0009834/   -> UBERON:0009834
    """
    parts = (
        df["FILE_biosample_ontology"]
        .str.split("/", expand=True)[2]
        .str.rsplit("_", n=2, expand=True)
    )

    return parts[1] + ":" + parts[2]

In [ ]:
encode_df[BIOSPECIMEN_ID_COL] = extract_biosample_curie(encode_df)

In [ ]:
epiatlas_df[BIOSPECIMEN_ID_COL] = epiatlas_df["harmonized_sample_ontology_curie"]

Verify there are no missing terms, and that they all conform to expected format.

In [ ]:
for df in [epiatlas_df, encode_df]:
    col = df[BIOSPECIMEN_ID_COL]
    display(col.head())
    assert col.isnull().sum() == 0
    assert col.apply(
        lambda s: any(source in s for source in biosample_ontology_sources)
    ).all()
    assert col.str.contains(":").all()

In [ ]:
display(encode_df[ASSAY].value_counts(dropna=False))

In [ ]:
encode_df["has_epiatlas_biospecimen_overlap"] = encode_df[BIOSPECIMEN_ID_COL].isin(
    epiatlas_df[BIOSPECIMEN_ID_COL]
)
overlap_count = encode_df["has_epiatlas_biospecimen_overlap"].sum()
print(
    f"ENCODE samples with exact EpiATLAS biospecimen overlap: {overlap_count} / {encode_df.shape[0]} ({100 * overlap_count / encode_df.shape[0]:.1f}%)"
)

In [ ]:
encode_df_core = encode_df[~encode_df[ASSAY].isin(["ctcf", "non-core"])]
display(encode_df_core[ASSAY].value_counts(dropna=False))

overlap_count = encode_df_core["has_epiatlas_biospecimen_overlap"].sum()
print(
    f"ENCODE core samples with exact EpiATLAS biospecimen overlap: {overlap_count} / {encode_df_core.shape[0]} ({100 * overlap_count / encode_df_core.shape[0]:.1f}%)"
)

## 4. R1 — Major comment 4: Classifier performance on SHAP-selected regions

**Reviewer comment (R1 Major 4; also R2 Question 3).** It would be informative to quantify the contribution of the SHAP-selected genomic bins to model performance — e.g. training using only these bins, or after masking them.

**Analysis.** We compare Assay and Biospecimen classifier performance across five 100 kb feature sets: the SHAP-derived intersection (118 regions) and union (4,509 regions) of important regions for the biologically relevant classifiers, size-matched random controls (`random_n118`, `random_n4510`), and the full feature set (`all_none`). This mirrors the UUID/EpiRR comparison above, reusing the same `obtain_all_feature_set_data` + `graph_feature_set_metrics` machinery. SHAP regions carry more information than random regions of the same size but with a small effect size, consistent with the high correlation between 100 kb bins (median pairwise absolute Pearson correlation of 0.616; see Appendix B). The published figure caption reports **uncorrected** two-sided Welch's t-tests. As an additional robustness check, `df_shap_focus` below restricts to the interpretable comparisons — each SHAP set versus its equal-N random control, plus the union set versus the full feature set — and applies Benjamini–Hochberg FDR correction, reporting per-fold mean differences (`mean_diff`) as the effect size. The SHAP-versus-random advantage stays significant after this correction, and the union is indistinguishable from the full feature set.

**Rebuttal → notebook** — each cited number or figure, and the cell/output that produces it:
- Rebuttal Figure 3 (Assay and Biospecimen performance per feature set) → the `graph_feature_set_metrics` figures.
- Two-sided Welch's t-tests → `df_tests_shap` (all pairs); focused robustness check with BH-FDR (`p_adj_bh`) and effect size (`mean_diff`) → `df_shap_focus`.
- Median pairwise |Pearson| correlation between 100 kb bins (0.616) → Appendix B.

In [ ]:
compare_SHAP = [
    "hg38_100kb_random_n118_none",
    "hg38_100kb_global_tasks_intersection_none",
    "hg38_100kb_random_n4510_none",
    "hg38_100kb_global_tasks_union_none",
    "hg38_100kb_all_none",
]

In [ ]:
shap_all_metrics = split_results_handler.obtain_all_feature_set_data(
    parent_folder=mixed_data_dir,
    merge_assays=True,
    return_type="metrics",
    include_sets=compare_SHAP,
    include_categories=[ASSAY, CELL_TYPE],
    exclude_names=["16ct", "27ct", "7c", "chip-seq-only"],
    verbose=False,
)

In [ ]:
# Order the metrics
shap_selected_metrics = {
    name: shap_all_metrics[name]  # type: ignore
    for name in compare_SHAP
    if name in shap_all_metrics
}

shap_selected_metrics = change_classifier_task_name(shap_selected_metrics)

In [ ]:
# NOTE: the manuscript figure caption reports the uncorrected two-sided Welch's
# t-tests (df_tests_shap). The table below is an additional robustness check.
# Focused comparisons for the SHAP-contribution claim: each SHAP set vs its equal-N
# random control (isolating selection from feature-set size), plus the union SHAP set
# vs the full feature set (how much signal the union retains). Benjamini-Hochberg FDR
# is applied across this restricted family; mean_diff = mean_b - mean_a is the per-fold
# effect size (read its direction from the feature_set_a / feature_set_b columns).
comparisons_of_interest = {
    frozenset(
        ("hg38_100kb_random_n118_none", "hg38_100kb_global_tasks_intersection_none")
    ),
    frozenset(("hg38_100kb_random_n4510_none", "hg38_100kb_global_tasks_union_none")),
    frozenset(("hg38_100kb_global_tasks_union_none", "hg38_100kb_all_none")),
}

df_tests_shap = pairwise_ttests(shap_selected_metrics)

df_shap_focus = df_tests_shap[
    df_tests_shap.apply(
        lambda r: frozenset((r["feature_set_a"], r["feature_set_b"]))
        in comparisons_of_interest,
        axis=1,
    )
].copy()

df_shap_focus["mean_diff"] = df_shap_focus["mean_b"] - df_shap_focus["mean_a"]
df_shap_focus["p_adj_bh"] = multipletests(df_shap_focus["p_value"], method="fdr_bh")[1]
df_shap_focus["significance_adj"] = df_shap_focus["p_adj_bh"].apply(
    lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
)

with pd.option_context(
    "display.max_rows", None, "display.max_columns", None, "display.width", None
):
    display(df_shap_focus)

In [ ]:
graph_feature_set_metrics(
    all_metrics=shap_selected_metrics,
    feature_set_order=[
        "hg38_100kb_random_n118_none",
        "hg38_100kb_global_tasks_intersection_none",
        "hg38_100kb_random_n4510_none",
        "hg38_100kb_global_tasks_union_none",
        "hg38_100kb_all_none",
    ],
    boxpoints="all",
    height=600,
)

## 5. R2: WGBS biospecimen-source coverage in the 16 trained classes

**Reviewer comment.** EpiClass performance on WGBS is not analyzed to the same depth as ChIP-seq and RNA-seq; could the authors apply it to ENCODE WGBS or explain why not?

**Context.** EpiClass was applied to ENCODE WGBS (five of six classifiers in Fig. 3A). The Biospecimen source classifier, however, could not be fairly evaluated on ENCODE WGBS because few evaluable samples fall within its 16 trained classes. This section quantifies that coverage (reusing `encode_df` / `epiatlas_df` from section 3).

**Analysis.** Using the ontology CURIEs of the 16 biospecimen classifier classes (from EpiATLAS), we count how many evaluable ENCODE WGBS samples fall within them.

**Rebuttal → notebook** — each cited number or figure, and the cell/output that produces it:
- 28 / 98 evaluable ENCODE WGBS samples within the 16 classes → the final WGBS print.

In [ ]:
EPIATLAS_16_CT = [
    ct.lower()
    for ct in [
        "T cell",
        "neutrophil",
        "brain",
        "monocyte",
        "lymphocyte of B lineage",
        "myeloid cell",
        "venous blood",
        "macrophage",
        "mesoderm-derived structure",
        "endoderm-derived structure",
        "colon",
        "connective tissue cell",
        "hepatocyte",
        "mammary gland epithelial cell",
        "muscle organ",
        "extraembryonic cell",
    ]
]

In [ ]:
print(epiatlas_df.shape)
epiatlas_16_ct_df = epiatlas_df[epiatlas_df[CELL_TYPE].str.lower().isin(EPIATLAS_16_CT)]
print(epiatlas_16_ct_df.shape)

In [ ]:
curie_16_ct = epiatlas_16_ct_df[BIOSPECIMEN_ID_COL].unique().tolist()

In [ ]:
encode_wgbs_df = encode_df[encode_df[ASSAY] == "wgbs"]
print(f"Number of ENCODE WGBS samples: {len(encode_wgbs_df)}")
sub_df = encode_wgbs_df[encode_wgbs_df[BIOSPECIMEN_ID_COL].isin(curie_16_ct)]
N = sub_df.shape[0]
print(
    f"Number of ENCODE WGBS samples within biospecimen source classifier 16 classes: {N}/{len(encode_wgbs_df)} ({N/len(encode_wgbs_df):.1%})\n"
)
print(f"Representing {sub_df[CELL_TYPE].nunique()}/{len(EPIATLAS_16_CT)} classes:\n")
print(sub_df[CELL_TYPE].value_counts(dropna=False))

## 6. R3 — Question 1: Per-assay vs mixed-assay biospecimen classifier

**Reviewer comment.** Have the authors considered establishing protocol-type-specific models?

**Context.** The Assay classifier already uses protocol-specific classes. For metadata categories such as biospecimen source we compare a single mixed-assay classifier against classifiers trained separately per assay. The mixed model's validation predictions are sliced per assay per fold and re-scored, so each per-assay `*_mixed` result is directly comparable (and t-testable) against the matching single-assay `*_only` classifier.

**Analysis.** Per-fold accuracy for `*_only` vs `*_mixed` per assay, plus one-sided Welch's t-tests (H1: per-assay > mixed).

**Rebuttal → notebook** — each cited number or figure, and the cell/output that produces it:
- Rebuttal Figure 4 (per-assay accuracy, only vs mixed) → the `NN_performance_per_assay` figure.
- No statistically significant per-assay improvement → `only_vs_mixed_ttest_df` (no p below 0.05).

In [ ]:
base_results_dir = base_data_dir / "training_results" / "dfreeze_v2"
results_dir_1 = (
    base_results_dir
    / "hg38_100kb_all_none"
    / "harmonized_sample_ontology_intermediate_1l_3000n"
    / "10fold-oversampling-unique_assay"
)

assay_results = {}
for assay_folder in results_dir_1.glob("*_only"):
    assay_results[assay_folder.name] = split_results_handler.read_split_results(
        assay_folder
    )

assay_metrics = split_results_handler.compute_split_metrics(
    assay_results, concat_first_level=True
)
assay_metrics = split_results_handler.invert_metrics_dict(assay_metrics)

In [ ]:
result_dir_mixed = (
    base_results_dir
    / "hg38_100kb_all_none"
    / "harmonized_sample_ontology_intermediate_1l_3000n"
    / "10fold-oversampling"
)

# md5sum-indexed metadata, with rna/wgbs sub-assays merged to match ASSAY_ORDER.
metadata_2_df = metadata_handler.load_metadata_df("v2")
assay_by_id = metadata_2_df[ASSAY]

mixed_split_dfs = split_results_handler.read_split_results(result_dir_mixed)

# Group each fold's predictions by assay via the metadata bridge, keeping the
# prediction columns ("True class", "Predicted class", per-class softmax) intact so
# compute_split_metrics sees the same schema as the *_only runs.
mixed_assay_results: Dict[str, Dict[str, pd.DataFrame]] = defaultdict(dict)
for split_name, split_df in mixed_split_dfs.items():
    split_assays = assay_by_id.reindex(split_df.index)
    for assay, assay_df in split_df.groupby(split_assays):
        mixed_assay_results[f"{assay}_mixed"][split_name] = assay_df

mixed_assay_metrics = split_results_handler.compute_split_metrics(
    mixed_assay_results, concat_first_level=True
)
mixed_assay_metrics = split_results_handler.invert_metrics_dict(mixed_assay_metrics)

# Fold the per-assay mixed metrics into assay_metrics next to the *_only entries.
assay_metrics.update(mixed_assay_metrics)

In [ ]:
def NN_performance_per_assay(
    assay_metrics: Dict[str, Dict[str, Dict[str, float]]],
    logdir: Optional[Path] = None,
    name: Optional[str] = None,
    title_end: str = "",
    y_range: None | List[float] = None,
):
    """Box plot of per-assay accuracy: single-assay (*_only) vs mixed (*_mixed)."""
    fig = go.Figure()
    for assay in ASSAY_ORDER:
        try:
            only_metrics = assay_metrics[f"{assay}_only"]
            mixed_metrics = assay_metrics[f"{assay}_mixed"]
        except KeyError:
            print(f"KeyError. Skipping '{assay}'")
            continue

        for suffix, metrics in (("only", only_metrics), ("mixed", mixed_metrics)):
            assay_acc = {split: metrics[split]["Accuracy"] for split in metrics}
            fig.add_trace(
                go.Box(
                    y=list(assay_acc.values()),
                    name=f"{assay}_{suffix}",
                    boxmean=True,
                    boxpoints="all",
                    showlegend=True,
                    marker=dict(size=3, color="black"),
                    line=dict(width=1, color="black"),
                    fillcolor=assay_colors[assay],
                    hovertemplate="%{text}",
                    text=[f"{split}: {value:.4f}" for split, value in assay_acc.items()],
                )
            )

    if y_range is not None:
        fig.update_yaxes(range=y_range)

    title_text = "NN classification (100kb_all_none) - Sample ontology - Training per assay VS mixed"
    if title_end:
        title_text += f" - {title_end}"
    fig.update_layout(
        title_text=title_text,
        yaxis_title="Accuracy",
        xaxis_title="Assay",
        width=1000,
        height=700,
    )

    # Save figure
    if logdir is not None and name is not None:
        save_figure(fig, logdir, name)

    fig.show()

In [ ]:
y_min = 0.5
NN_performance_per_assay(
    assay_metrics=assay_metrics,
    y_range=[y_min, 1.001],
)

### Welch's t-test, per assay: `*_only` vs `*_mixed`

In [ ]:
def only_vs_mixed_ttests(
    assay_metrics: Dict[str, Dict[str, Dict[str, float]]],
    assays: List[str] | None = None,
    k: int = 10,
    alternative: str = "greater",
) -> pd.DataFrame:
    """Welch's t-test (over k folds) of the single-assay vs mixed classifier.

    Calls ``pairwise_ttests`` once per assay on the single ``*_only`` / ``*_mixed``
    pair, wrapping each feature set under the assay name as the (single) category so
    it matches the {feature_set: {category: {split: metric_dict}}} format expected.

    The pair is ordered (``*_only``, ``*_mixed``), so the default
    ``alternative="greater"`` tests H1: mean(only) > mean(mixed); a significant
    result flags an assay where the single-assay classifier beats the mixed one.
    """
    if assays is None:
        assays = ASSAY_ORDER
    results = []
    for assay in assays:
        only_key, mixed_key = f"{assay}_only", f"{assay}_mixed"
        if only_key not in assay_metrics or mixed_key not in assay_metrics:
            print(f"Missing _only/_mixed metrics for '{assay}'. Skipping.")
            continue
        pair_metrics = {
            only_key: {assay: assay_metrics[only_key]},
            mixed_key: {assay: assay_metrics[mixed_key]},
        }
        results.append(pairwise_ttests(pair_metrics, k=k, alternative=alternative))
    return pd.concat(results, ignore_index=True)

In [ ]:
only_vs_mixed_ttest_df = only_vs_mixed_ttests(assay_metrics)
display(only_vs_mixed_ttest_df)

In [ ]:
print("Smallest 5 p-values")
display(only_vs_mixed_ttest_df.sort_values(by="p_value").head(n=5))

## Appendix A — Biospecimen performance by consortium (internal review comment)

This section addresses an internal review comment on the flagship paper (project/lab as a possible confounder for biospecimen prediction, analogous to the biomaterial-type comment from R1) rather than an official EpiClass reviewer comment. It is kept separate for that reason.

**Comment.** Biospecimen prediction on ChIP-seq input controls is ~75% (100 kb bins). Is it still this high when conditioned on consortium? If not, that could indicate batch effects drive it.

**Analysis.** We break biospecimen classifier performance down by consortium (project), on all assays and on the input-only subset, against a naive baseline that predicts the majority biospecimen class within each consortium — the ceiling of a pure "consortium identity" predictor. EpiClass exceeding this baseline where multiple biospecimen classes coexist within a consortium indicates biospecimen-discriminative signal beyond consortium fingerprints. Folds where a consortium has a single biospecimen class are uninformative (any classifier trivially scores 100%).

In [ ]:
data_dir = data_dir_100kb / f"{CELL_TYPE}_1l_3000n" / "10fold-oversampling"
if not data_dir.exists():
    raise FileNotFoundError(f"Directory {data_dir} does not exist.")

In [ ]:
cell_type_results = split_results_handler.obtain_all_feature_set_data(
    parent_folder=gen_data_dir,
    merge_assays=True,
    return_type="split_results",
    include_categories=[CELL_TYPE],
    include_sets=["hg38_100kb_all_none"],
    verbose=False,
)

cell_type_split_dfs = cell_type_results["hg38_100kb_all_none"][CELL_TYPE]

In [ ]:
input_mask = metadata_v2_df[ASSAY] == "input"
input_md5s = set(metadata_v2_df.loc[input_mask].index)

In [ ]:
cell_type_results_input = {}
for split_name, split_df in cell_type_split_dfs.items():
    print(f"Processing split: {split_name}")
    print(f"  Total samples in split: {len(split_df)}")
    input_df = split_df[split_df.index.isin(input_md5s)]
    print(f"  Samples with 'input' assay in split: {len(input_df)}")
    if len(input_df) == 0:
        print(f"  WARNING: No input samples in {split_name}")
        continue
    cell_type_results_input[split_name] = input_df

In [ ]:
cell_type_metrics_breakdown = prepare_metrics_per_category(
    split_dfs=cell_type_split_dfs,  # type: ignore
    breakdown_col="project",
    verbose=True,
)

In [ ]:
cell_type_metrics_breakdown_input = prepare_metrics_per_category(
    split_dfs=cell_type_results_input,  # type: ignore
    breakdown_col="project",
    verbose=False,
)

In [ ]:
naive_metrics_df = compute_naive_majority_metrics_per_category(
    split_dfs=cell_type_split_dfs,  # type: ignore
    breakdown_col="project",
    verbose=False,
)

In [ ]:
naive_metrics_df_input = compute_naive_majority_metrics_per_category(
    split_dfs=cell_type_results_input,  # type: ignore
    breakdown_col="project",
    verbose=False,
)

In [ ]:
label_remapper = {
    "NIH Roadmap Epigenomics": "Roadmap",
    "Korea Epigenome Project (KNIH)": "KNIH",
}

In [ ]:
# all assays
plot_metrics_per_category(
    metrics_per_category=cell_type_metrics_breakdown,
    category_label="Project",
    naive_metrics_df=naive_metrics_df,
    naive_category_col="project",
    metrics_to_plot=["Accuracy", "F1_macro"],
    xaxis_label_map=label_remapper,
    # logdir=Path.home() / "downloads",
    # filename_prefix="cell_type_metrics_per_project",
)

In [ ]:
# input only
plot_metrics_per_category(
    metrics_per_category=cell_type_metrics_breakdown_input,
    category_label="Project",
    naive_metrics_df=naive_metrics_df_input,
    naive_category_col="project",
    metrics_to_plot=["Accuracy", "F1_macro"],
    xaxis_label_map=label_remapper,
    # logdir=Path.home() / "downloads",
    # filename_prefix="cell_type_metrics_per_project_input_only",
)

## Appendix B — Bin–bin correlation at 100 kb (median pairwise |Pearson| = 0.616)

The discussion statement that 100 kb bins are highly correlated (median pairwise absolute Pearson correlation of 0.616, cited in the Section 4 / R1 Major 4 response) is computed from the raw signal matrix, not the prediction CSVs used elsewhere in this notebook. It loads a ~2 GB signal `.npz` and builds a 30,321 × 30,321 correlation matrix (~7 GB RAM), so rather than run it inline it is documented here; it was executed once against the training signal matrix and returned 0.616:

```python
import numpy as np
from pathlib import Path

npz_path = Path.home() / "Projects/epiclass/input/hdf5/epiatlas_dfreeze_100kb_all_none.npz"
list_path = (
    Path.home()
    / "Projects/epiclass/input/hdf5_list/hg38_epiatlas-freeze-v2"
    / "100kb_all_none_dfreeze_filterCtl_plus_4ctl.list"
)

with np.load(npz_path) as data:
    matrix, ids = data["signals"], data["ids"]

# keep the signal files used for training (filtered controls + 4 controls)
md5s = [Path(p).stem.split("_")[0] for p in list_path.read_text().splitlines()]
filtered = matrix[np.isin(ids, md5s)]          # (20922 n_files, 30_321 bins)

corr = np.corrcoef(filtered, rowvar=False)      # 30_321 x 30_321
upper = corr[np.triu_indices(corr.shape[0], k=1)]
upper = upper[~np.isnan(upper)]
print(np.median(np.abs(upper)))                 # -> 0.616
```

This high inter-bin correlation is the most likely reason the SHAP-selected regions beat size-matched random controls by only a small margin (Section 4): predictive signal is spread across many correlated bins, so no compact region set is strongly favoured.